# M3 评测 v21（Colab + AI Pro）—— 16k checkpoint × 10 任务 × 10 episodes

- GPU: 运行类型选 **T4 GPU**
- 断点续测: 每个任务(~20min)完成即上传 HF，断开后重跑自动跳过已完成任务
- 需要: HF_TOKEN（userdata secret 或运行时手动输入）

In [ ]:
# ===== v21.4 Colab 评测：watcher 每 3 分钟上传 partial 日志（任务内验尸数据） =====
# 断点续测：每个任务(~20min)完成即上传 HF；会话断开后重跑自动跳过已完成任务
import os, re, glob, json, time, shutil, subprocess, urllib.request

W = "/content"
TOPIC = "smolvla2-ep-Kp9wQ4tRz7"
REPO = "A2bao/smolvla-libero-spatial"
CODE_SHA = "146d8474932c8b566717eb2738ba459337ab71c4"
REV = "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4"
os.environ["HF_HOME"] = W + "/hf_cache"
os.makedirs(W + "/logs", exist_ok=True)

def run(cmd, timeout=3600):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def beacon(msg):
    try:
        urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + TOPIC,
                data=str(msg).encode()), timeout=8)
    except Exception as e:
        print("[beacon fail]", repr(e), flush=True)

# ---- 0) HF token ----
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("[0] token from userdata", flush=True)
except Exception:
    os.environ["HF_TOKEN"] = input("粘贴 HF_TOKEN（hf_ 开头）: ").strip()

# ---- 1) lerobot 安装（固定 commit）+ dtype 补丁 ----
def have_lerobot():
    return "0.6.2" in run("python -c 'import lerobot; print(lerobot.__version__)'")
if not have_lerobot():
    for cmd in ('pip install -q "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"',):
        print("[1] install ...", flush=True)
        print(run(cmd, timeout=2400)[-500:], flush=True)
        if have_lerobot():
            break
    if not have_lerobot():
        beacon("FAIL-install")
        raise RuntimeError("lerobot 安装失败")
SITE = run("python -c 'import site; print(site.getsitepackages()[0])'")
run("sed -i 's/torch_dtype=\"bfloat16\"/torch_dtype=\"float32\"/' " + SITE + "/lerobot/policies/smolvla/smolvlm_with_expert.py")
ok = "float32" in run("grep torch_dtype " + SITE + "/lerobot/policies/smolvla/smolvlm_with_expert.py")
print("[1] lerobot 0.6.2 + dtype patch:", ok, flush=True)
assert ok, "dtype 补丁失败"
beacon("OK-install")

# Colab 预装环境冲突：老 torchao 缺 FqnToConfig，与 lerobot 依赖链的新 diffusers 冲突
# （导入链经过未使用的 fastwam/wan 策略）。本评测不需要量化：卸载 torchao + 锁定兼容 diffusers
run("pip uninstall -q -y torchao 2>&1 | tail -1")
run('pip install -q "diffusers<0.36" 2>&1 | tail -1')

# ---- 2) 从 HF 逐文件拉取 16k checkpoint（pretrained_model 含 tokenizer 子目录）----
from huggingface_hub import list_repo_files, hf_hub_download, upload_file
_files = [f for f in list_repo_files(REPO, repo_type="model", token=os.environ["HF_TOKEN"])
          if f.startswith("checkpoints-backup/0008000/")]
assert _files, "HF 上没有 0008000 备份"
for f in _files:
    hf_hub_download(REPO, f, repo_type="model", token=os.environ["HF_TOKEN"], local_dir=W + "/ckpt_dl")
CKPT = W + "/ckpt_dl/checkpoints-backup/0008000/pretrained_model"
for f in ("model.safetensors", "policy_preprocessor.json", "train_config.json", "tokenizer/tokenizer_config.json"):
    assert os.path.exists(CKPT + "/" + f), "checkpoint 缺文件: " + f
print("[2] checkpoint:", CKPT, flush=True)
beacon("OK-ckpt")

# ---- 3) libero assets + 配置 ----
ASSETS = """import time
from pathlib import Path
from huggingface_hub import snapshot_download
import importlib.util
home = Path.home()
t0 = time.time()
snapshot_download(repo_id='lerobot/libero-assets', repo_type='dataset', local_dir='/content/libero_assets')
spec = importlib.util.find_spec('libero')
lib_dir = Path(spec.origin).parent / 'libero'
(home / '.libero').mkdir(exist_ok=True)
lines = [
    'assets: /content/libero_assets',
    'bddl_files: %s/bddl_files' % lib_dir,
    'datasets: %s/../datasets' % lib_dir.parent,
    'init_states: %s/init_files' % lib_dir,
]
(home / '.libero' / 'config.yaml').write_text(chr(10).join(lines) + chr(10))
print('assets %.0fs' % (time.time() - t0))
"""
open("/tmp/assets.py", "w").write(ASSETS)
out = run("python /tmp/assets.py", timeout=1800)
print("[3] assets:", out[-120:], flush=True)

# ---- 4) 数据集缓存（pinned revision）----
open("/tmp/dsfix.py", "w").write(
    "from huggingface_hub import snapshot_download\n"
    "snapshot_download('lerobot/libero', repo_type='dataset', revision='" + REV + "', max_workers=8)\n"
    "print('DS_OK')\n")
for attempt in range(3):
    out = run("python /tmp/dsfix.py 2>&1 | tail -2", timeout=2400)
    if "DS_OK" in out:
        break
    print("[4] 数据集重试", attempt + 1, ":", out[-200:], flush=True)
else:
    beacon("FAIL-dataset")
    raise RuntimeError("数据集缓存失败: " + out[-300:])
print("[4] 数据集就绪", flush=True)
beacon("OK-setup")

# ---- 5) 按任务断点续测：10 个任务，每个 10 episodes ≈ 20min，完成即上传 ----
def hf_list():
    try:
        return list_repo_files(REPO, repo_type="model", token=os.environ["HF_TOKEN"])
    except Exception:
        return []

def hf_up(local, remote):
    try:
        upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=REPO,
                    repo_type="model", token=os.environ["HF_TOKEN"])
        print("✓ 上传", remote, flush=True)
        return True
    except Exception as e:
        print("✗ 上传失败", remote, repr(e)[:100], flush=True)
        return False

done_tasks = {int(f.split("/")[1][4:]) for f in hf_list()   # "task3" -> 3
              if f.startswith("eval16k/task") and f.endswith("eval_info.json")}
print("[5] 已完成任务:", sorted(done_tasks), flush=True)

ENV = ("export MUJOCO_GL=egl; export HF_HOME=" + W + "/hf_cache; export PYTHONUNBUFFERED=1;")
rates = {}
for tid in range(10):
    if tid in done_tasks:
        print("[5] task", tid, "已完成，跳过", flush=True)
        continue
    OUT = W + "/eval/task%d" % tid
    LOG = W + "/logs/eval_t%d.log" % tid
    os.makedirs(LOG.rsplit("/", 1)[0], exist_ok=True)
    open(LOG, "w").close()
    _flag = W + "/logs/t%d.done" % tid
    if os.path.exists(_flag): os.remove(_flag)
    _tick = [0]
    def _watch(log=LOG, flag=_flag, tid=tid):
        while not os.path.exists(flag):
            time.sleep(90)
            _tick[0] += 1
            try:
                raw = open(log, errors="ignore").read().replace("\r", "\n")
                eps = len(re.findall(r"Running rollout", raw))
                ls = [l for l in raw.split("\n") if l.strip()]
                beacon("T%d EP%d | %s" % (tid, eps, ls[-1][-80:] if ls else ""))
                if _tick[0] % 2 == 0:   # 每 ~3 分钟把 partial 日志增量上传 HF（死后可验尸）
                    run("gzip -cf " + log)
                    hf_up(log + ".gz", "eval16k/task%d/partial_eval.log.gz" % tid)
            except Exception:
                pass
    import threading
    threading.Thread(target=_watch, daemon=True).start()
    CMD = ("lerobot-eval --policy.path=" + CKPT + " "
           "--env.type=libero --env.task=libero_spatial --env.task_ids=\"[%d]\" " % tid +
           "--env.observation_height=256 --env.observation_width=256 "
           "--env.init_states=true --env.hard_reset=true "
           "--policy.n_action_steps=1 "
           "--eval.n_episodes=10 --eval.batch_size=1 "
           "--output_dir=" + OUT + " --policy.push_to_hub=false")
    print("[5] 任务 %d 评测中（进度看 ntfy 信标）...", flush=True)
    ret = subprocess.call("bash -c '" + ENV + " " + CMD + " > " + LOG + " 2>&1'", shell=True)
    open(_flag, "w").close()
    infos = glob.glob(OUT + "/**/eval_info.json", recursive=True)
    if ret == 0 and infos:
        d = json.load(open(infos[0]))
        suc = d["per_task"][0]["metrics"]["successes"]
        rate = sum(bool(s) for s in suc) / len(suc)
        rates[tid] = (rate, len(suc))
        hf_up(infos[0], "eval16k/task%d/eval_info.json" % tid)
        run("gzip -cf " + LOG + " > " + LOG + ".gz")
        hf_up(LOG + ".gz", "eval16k/task%d/eval.log.gz" % tid)
        vid = glob.glob(OUT + "/**/eval_episode_0.mp4", recursive=True)
        if vid: hf_up(vid[0], "eval16k/videos/task%d_ep0.mp4" % tid)
        beacon("TASK%d done %d/10 (累计完成 %d/10 任务)" % (tid, round(rate*10), len(rates)+len(done_tasks)))
        print("[5] task %d: %d/10 成功" % (tid, round(rate*10)), flush=True)
    else:
        tail = run("tail -c 1500 " + LOG + " | tr '\\r' '\n'")
        print(tail[-1000:], flush=True)
        beacon("TASK%d FAIL ret=%d" % (tid, ret))

# ---- 6) 汇总 ----
total_succ = total_n = 0
for tid in range(10):
    infos = glob.glob(W + "/eval/task%d/**/eval_info.json" % tid, recursive=True)
    if not infos: continue
    d = json.load(open(infos[0]))
    suc = d["per_task"][0]["metrics"]["successes"]
    total_succ += sum(bool(s) for s in suc); total_n += len(suc)
if total_n:
    msg = "EVAL-DONE 16k: %d/%d = %.0f%%" % (total_succ, total_n, 100.0*total_succ/total_n)
    print("[6]", msg, flush=True)
    beacon(msg)
    agg = {"total_successes": total_succ, "total_episodes": total_n,
           "success_rate": total_succ/total_n, "true_updates": 16000}
    open(W + "/aggregate.json", "w").write(json.dumps(agg))
    hf_up(W + "/aggregate.json", "eval16k/aggregate.json")
else:
    beacon("EVAL-DONE 无新完成任务（可能全部已完成）")
print("ALL DONE", flush=True)
